In [4]:
import json
import tiktoken

def extract_think(text: str) -> str:
    if "think" in text:
        return text.split("<think>")[-1].split("</think>")[0].strip()
    elif "<think>" in text:
        return text.split("<think>")[-1].split("</think>")[0].strip()
    else:
        return text.split("<think>")[-1].split("</think>")[0].strip()

def count_tokens(text: str) -> int:
    encoding = tiktoken.get_encoding("cl100k_base")
    tokens = encoding.encode(text)
    return len(tokens)

In [5]:
models = [
    "DeepSeek-V3",
    "DeepSeek-R1",
    "Qwen2.5-72B-Ins",
    "QwQ-32B",
]



In [ ]:
for model in models:
    # print("==" * 10 + f" {model} " + "==" * 10)
    filepath1 = f"results/codegeneration/{model}/lcb_generation.json"
    filepath2 = f"results/codeexecution/{model}/lcb_output.json"
    filepath3 = f"results/codeexecution/{model}/crux_output_record.json"
    
    output_text = f"{model} & "
    for filepath in [filepath1, filepath2, filepath3]:
        
        if "lcb" in filepath:
            with open(filepath.replace(".json", "_eval.json"), "r") as f:
                data = json.load(f)
                score = data[0]["pass@1"] if "output" in filepath else data[0]["pass@1"] * 100
        else:
            with open(filepath.replace("_record", "_scored"), "r") as f:
                data = json.load(f)
                score = data["pass_at_1"]
        
        with open(filepath, "r") as f:
            data = json.load(f)
        total_tokens = 0
        max_tokens = 0
        for entry in data:
            # print(entry)
            token_count = count_tokens(extract_think(entry["output_list"][0]))
            total_tokens += token_count
            max_tokens = max(max_tokens, token_count)
        
        avg_tokens = total_tokens / len(data)
        output_text += f"${score:.2f}$ & ${avg_tokens:.0f}$ & ${max_tokens:.0f}$ & "
    output_text = output_text[:-2] + "\\\\"
    print(output_text)


DeepSeek-V3 & $76.32$ & $722$ & $1690$ & $92.07$ & $428$ & $1938$ & $88.00$ & $272$ & $1856$ \\
DeepSeek-R1 & $81.60$ & $3169$ & $25887$ & $98.96$ & $895$ & $4987$ & $92.62$ & $903$ & $5012$ \\
Qwen2.5-72B-Ins & $44.42$ & $240$ & $680$ & $90.40$ & $491$ & $1942$ & $78.25$ & $295$ & $995$ \\
QwQ-32B & $86.30$ & $6283$ & $32693$ & $98.96$ & $1518$ & $9206$ & $93.25$ & $1391$ & $13718$ \\


In [ ]:
models = [
    "DeepSeek-V3",
    "DeepSeek-R1",
    "Qwen2.5-72B-Ins",
    "QwQ-32B",
]



difficulty = {
    "easy": [],
    "medium": [],
    "hard": [],
}

for model in models:
    # print("==" * 10 + f" {model} " + "==" * 10)
    filepath1 = f"results/codegeneration/{model}/lcb_generation.json"
    filepath2 = f"results/codeexecution/{model}/lcb_output.json"
    
    output_text = f"{model} & "
    for filepath in [filepath1, filepath2]:
        
        with open(filepath.replace(".json", "_eval.json"), "r") as f:
            data = json.load(f)
            score = data[0]["pass@1"] if "output" in filepath else data[0]["pass@1"] * 100
        
        with open(filepath, "r") as f:
            data = json.load(f)
        total_tokens = 0
        max_tokens = 0
        
        for entry in data:
            token_count = count_tokens(extract_think(entry["output_list"][0]))
            difficulty[entry["difficulty"]].append(token_count)
            
            # gpt_4o_metrc score
            # total_tokens += token_count
            # max_tokens = max(max_tokens, token_count)
        
        for key in difficulty:
            total_tokens = sum(difficulty[key])
            max_tokens = max(difficulty[key])
            avg_tokens = total_tokens / len(difficulty[key])
            
            print(f"difficulty: {key} {model} - {avg_tokens:.0f} \\\\")
        
        print("=" * 20)
            


difficulty: easy DeepSeek-V3 - 556 \\
difficulty: medium DeepSeek-V3 - 772 \\
difficulty: hard DeepSeek-V3 - 884 \\
difficulty: easy DeepSeek-V3 - 447 \\
difficulty: medium DeepSeek-V3 - 615 \\
difficulty: hard DeepSeek-V3 - 858 \\
difficulty: easy DeepSeek-R1 - 747 \\
difficulty: medium DeepSeek-R1 - 1494 \\
difficulty: hard DeepSeek-R1 - 2999 \\
difficulty: easy DeepSeek-R1 - 709 \\
difficulty: medium DeepSeek-R1 - 1392 \\
difficulty: hard DeepSeek-R1 - 2941 \\
difficulty: easy Qwen2.5-72B-Ins - 613 \\
difficulty: medium Qwen2.5-72B-Ins - 1185 \\
difficulty: hard Qwen2.5-72B-Ins - 2095 \\
difficulty: easy Qwen2.5-72B-Ins - 572 \\
difficulty: medium Qwen2.5-72B-Ins - 1071 \\
difficulty: hard Qwen2.5-72B-Ins - 2067 \\
difficulty: easy QwQ-32B - 821 \\
difficulty: medium QwQ-32B - 1760 \\
difficulty: hard QwQ-32B - 4374 \\
difficulty: easy QwQ-32B - 846 \\
difficulty: medium QwQ-32B - 1780 \\
difficulty: hard QwQ-32B - 4348 \\


In [13]:
import json
with open("results/codeexecution/QwQ-32B/demo.json", "r") as f:
    data = json.load(f)

total_token = 0
total_useful_token = 0

for entry in data:
    total_token += count_tokens(extract_think(entry["output_list"][0]))
    total_useful_token += count_tokens(entry["useful_content"])
    
print(total_token / len(data))
print(total_useful_token / len(data))


1518.3423799582463
311.3883089770355
